# 1b — BATERIA overnight (HF puro, sem Unsloth) · Colab

**Por que sem Unsloth:** em 11/jun o Unsloth 2026.6.7 gerou um forward de CSM
quebrado (`audio_tokens_offsets` mismatch) com transformers 4.52.3 — bug do
ecossistema Unsloth desta semana, não nosso. Migrado pro **caminho oficial da
HuggingFace** (CsmForConditionalGeneration + peft): testado pela HF, estável,
e a A100-80GB tem VRAM de sobra sem a otimização do Unsloth.

**Modo "rodar 1x, voltar em ~3h".** PREFLIGHT (1 step em ~3min) antes de gastar
horas + time-budget global + checkpoints no Drive + try/except por experimento.

**Pergunta de ouro:** leitura limpa (CML-TTS, Frederico) vs podcast espontâneo
(TAGARELA) vs mix — qual ensina pt ao CSM-1B com menor WER?
**Resultado:** `Drive/TTS-ptbr-data/runs/BATERIA_results.md` + adapters/amostras.

In [ ]:
# ⚙️ CONFIG
TIME_BUDGET_MIN = 160
PER_EXP_MIN     = 50
LORA_R, LORA_ALPHA, LR = 64, 64, 5e-5
BATCH, ACCUM = 2, 16          # efetivo 32 (conservador; sobe depois que validar)

BATTERY = [
    {'name': 'A1_cml',      'source': 'cml',      'hours': 30},
    {'name': 'A3_tagarela', 'source': 'tagarela', 'hours': 25},
    {'name': 'A2_mix',      'source': 'mix',      'hours': 40},
]

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN'); os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr
DRIVE = '/content/drive/MyDrive/TTS-ptbr-data'; os.makedirs(f'{DRIVE}/runs', exist_ok=True)

In [ ]:
%%capture
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'
# transformers 4.52.3 = versão EXATA do checkpoint csm-1b (transformers 5.x renomeia
# pesos → "embed_audio_tokens MISSING"). torchao do Colab (0.10) é rejeitado pelo peft
# (quer >0.16) e não usamos → remover. HF puro, sem unsloth.
!pip install "transformers==4.52.3" peft accelerate "datasets>=3.4.1,<4.0.0" \
    soundfile jiwer librosa soxr bitsandbytes torchcodec faster-whisper==1.1.0 hf_transfer
!pip uninstall -y torchao

## Helpers (loaders · preprocess · CSMTrainer · eval WER · PREFLIGHT)

In [ ]:
from datasets import load_dataset, Audio, Dataset, concatenate_datasets
from transformers import AutoProcessor, CsmForConditionalGeneration, TrainingArguments, Trainer, TrainerCallback
from peft import LoraConfig, get_peft_model
import numpy as np, hashlib, time, gc, json, pathlib, torch

MODEL_ID = 'unsloth/csm-1b'   # mirror Apache ungated (mesmos pesos do sesame/csm-1b)
BF16 = torch.cuda.is_bf16_supported()

def load_source(source, hours):
    if source == 'cml':
        ds = load_dataset('ylacombe/cml-tts', 'portuguese', split='train')
    elif source == 'mix':
        cml = load_dataset('ylacombe/cml-tts', 'portuguese', split='train')
        mls = load_dataset('facebook/multilingual_librispeech', 'portuguese', split='train')
        norm = []
        for p in [cml, mls]:
            tcol = 'text' if 'text' in p.column_names else ('transcript' if 'transcript' in p.column_names else 'sentence')
            if tcol != 'text': p = p.rename_column(tcol, 'text')
            norm.append(p.remove_columns([c for c in p.column_names if c not in ('audio','text')]))
        ds = concatenate_datasets(norm)
    elif source == 'tagarela':
        st = load_dataset('freds0/TAGARELA', split='train', streaming=True)
        rows, tot, tgt = [], 0.0, hours*3600
        for ex in st:
            a = ex['audio']; rows.append({'audio': a, 'text': ex['sentence']})
            tot += len(a['array'])/a['sampling_rate']
            if tot >= tgt: break
        ds = Dataset.from_list(rows)
    ds = ds.cast_column('audio', Audio(sampling_rate=24000)).shuffle(seed=42)
    if source != 'tagarela':
        idx, tot = [], 0.0
        for i, ex in enumerate(ds):
            tot += len(ex['audio']['array'])/24000; idx.append(i)
            if tot >= hours*3600: break
        ds = ds.select(idx)
    return ds

def build_prep(processor, max_audio):
    def spk(ex, i):
        return str(int(hashlib.md5(str(ex.get('speaker_id', i)).encode()).hexdigest(), 16) % 10)
    def prep(ex, idx):
        conv = [{'role': spk(ex, idx), 'content': [{'type':'text','text':str(ex['text']).strip()},
                                                   {'type':'audio','path':ex['audio']['array']}]}]
        o = processor.apply_chat_template(conv, tokenize=True, return_dict=True, output_labels=True,
            text_kwargs={'padding':'max_length','max_length':256,'pad_to_multiple_of':8,'padding_side':'right'},
            audio_kwargs={'sampling_rate':24000,'max_length':max_audio,'padding':'max_length'},
            common_kwargs={'return_tensors':'pt'})
        return {k: v[0] for k, v in o.items()}
    return prep

def load_csm():
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model, _info = CsmForConditionalGeneration.from_pretrained(
        MODEL_ID, output_loading_info=True)  # float32: o codec Mimi gera float32, então o modelo
    # NÃO pode ser bf16 (mismatch no merge). bf16=True no Trainer faz autocast.
    _crit = [k for k in _info.get('missing_keys', []) if 'embed_audio' in k]
    assert not _crit, f'❌ pesos de ÁUDIO faltando no load ({_crit}) — transformers incompatível com o checkpoint'
    model = model.to('cuda')
    model.train(); model.codec_model.eval()    # codec Mimi congelado (exemplo oficial HF)
    return model, processor

def add_lora(model, r, alpha):
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

class CSMTrainer(Trainer):   # garante o codec Mimi sempre em eval (não treina)
    def training_step(self, model, inputs, *args, **kwargs):
        bm = model.get_base_model() if hasattr(model, 'get_base_model') else model
        if hasattr(bm, 'codec_model'): bm.codec_model.eval()
        return super().training_step(model, inputs, *args, **kwargs)

def eval_wer(model, processor, ref, out):
    import soundfile as sf, jiwer
    from faster_whisper import WhisperModel
    model.eval()
    bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
    gd = pathlib.Path(f'{out}/gen'); gd.mkdir(exist_ok=True, parents=True)
    for i, it in enumerate(bench):
        conv = [{'role':'0','content':[{'type':'text','text':str(ref['text'])},{'type':'audio','path':ref['audio']['array']}]},
                {'role':'0','content':[{'type':'text','text':it['text']}]}]
        inp = processor.apply_chat_template(conv, tokenize=True, return_dict=True).to('cuda')
        with torch.no_grad():
            au = model.generate(**inp, output_audio=True, max_new_tokens=375)
        sf.write(gd / f"{it.get('id', i)}.wav", au[0].to(torch.float32).cpu().numpy(), 24000)
    asr = WhisperModel('small', device='cpu', compute_type='int8')  # cpu: evita crash cuDNN no Colab
    norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(), jiwer.RemoveMultipleSpaces(), jiwer.Strip()])
    ws = []
    for i, it in enumerate(bench):
        segs, _ = asr.transcribe(str(gd / f"{it.get('id', i)}.wav"), language='pt')
        hyp = ' '.join(s.text.strip() for s in segs).strip()
        ws.append(jiwer.wer(norm(it['text']), norm(hyp)) if hyp else 1.0)
    del asr; gc.collect(); torch.cuda.empty_cache()
    model.train()
    return round(float(np.mean(ws)), 3)

def preflight():
    print('🔍 PREFLIGHT — 4 exemplos, 1 step de treino (HF puro)…')
    st = load_dataset('ylacombe/cml-tts', 'portuguese', split='train', streaming=True)
    rows = []
    for ex in st:
        rows.append({'audio': ex['audio'], 'text': ex['text']})
        if len(rows) >= 4: break
    raw = Dataset.from_list(rows).cast_column('audio', Audio(sampling_rate=24000))
    max_audio = min(20*24000+1, int(max(len(e['audio']['array']) for e in raw)) + 1)
    model, processor = load_csm()
    ds = raw.map(build_prep(processor, max_audio), with_indices=True, remove_columns=raw.column_names)
    model = add_lora(model, 8, 16)
    tr = CSMTrainer(model=model, train_dataset=ds, args=TrainingArguments(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=1, max_steps=1,
        bf16=BF16, fp16=not BF16, logging_steps=1, optim='adamw_8bit',
        output_dir='/tmp/preflight', report_to='none', remove_unused_columns=False))
    tr.train()
    # valida também a GERAÇÃO (o eval usa isso; senão só quebraria no fim das horas)
    import soundfile as _sf
    model.eval()
    _conv = [{'role':'0','content':[{'type':'text','text':str(raw[0]['text'])},{'type':'audio','path':raw[0]['audio']['array']}]},
             {'role':'0','content':[{'type':'text','text':'Teste rápido de geração.'}]}]
    _inp = processor.apply_chat_template(_conv, tokenize=True, return_dict=True).to('cuda')
    with torch.no_grad():
        _au = model.generate(**_inp, output_audio=True, max_new_tokens=64)
    assert _au[0].shape[-1] > 0, 'geração retornou vazio'
    del model, processor, tr, ds, raw; gc.collect(); torch.cuda.empty_cache()
    print('✅ PREFLIGHT PASSOU — treino E geração OK. A bateria pode rodar segura.')

def run_experiment(exp, deadline_global):
    name, out = exp['name'], f"{DRIVE}/runs/battery_{exp['name']}"
    os.makedirs(out, exist_ok=True); t0 = time.time()
    print(f"\n{'='*64}\n▶ {name}  ({exp['source']}, {exp['hours']}h)  {time.strftime('%H:%M')}\n{'='*64}")
    raw = load_source(exp['source'], exp['hours'])
    raw = raw.filter(lambda ex: 1.5 <= len(ex['audio']['array'])/24000 <= 20 and len(str(ex['text']).split()) >= 3)
    probe = raw.select(range(min(1500, len(raw))))
    MAX_AUDIO = min(20*24000+1, int(max(len(ex['audio']['array']) for ex in probe)) + 1)
    print(f"  {len(raw)} clipes · max_audio={MAX_AUDIO/24000:.0f}s")
    model, processor = load_csm()
    ds = raw.map(build_prep(processor, MAX_AUDIO), with_indices=True, remove_columns=raw.column_names, desc='tok')
    model = add_lora(model, LORA_R, LORA_ALPHA)
    cap_min = min(exp.get('minutes', PER_EXP_MIN), (deadline_global - time.time())/60 - 8)
    if cap_min < 5:
        print("  ⏱ sem tempo — pulando"); del model, processor, ds, raw; gc.collect(); torch.cuda.empty_cache(); return None
    print(f"  batch {BATCH}×{ACCUM} (efetivo {BATCH*ACCUM}) · cap {cap_min:.0f}min")
    class TimeCap(TrainerCallback):
        def __init__(s, m): s.dl = time.time() + m*60
        def on_step_end(s, a, st, c, **k):
            if time.time() > s.dl: c.should_training_stop = True
            return c
    tr = CSMTrainer(model=model, train_dataset=ds, args=TrainingArguments(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        num_train_epochs=99, learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.03,
        bf16=BF16, fp16=not BF16, logging_steps=10, optim='adamw_8bit', weight_decay=0.01,
        seed=3407, output_dir=out, report_to='none', save_steps=100, save_total_limit=1,
        remove_unused_columns=False), callbacks=[TimeCap(cap_min)])
    tr.train()
    steps = tr.state.global_step
    model.save_pretrained(f'{out}/final'); processor.save_pretrained(f'{out}/final')
    wer = eval_wer(model, processor, raw[0], out)
    r = {'name': name, 'source': exp['source'], 'hours': exp['hours'],
         'steps': steps, 'wer': wer, 'min': round((time.time()-t0)/60)}
    del model, processor, tr, ds, raw; gc.collect(); torch.cuda.empty_cache()
    print("  ✅", r); return r

## ▶️ Rodar — PREFLIGHT primeiro; só solta a bateria se passar

In [ ]:
import time, traceback
try:
    preflight(); SAFE = True
except Exception as e:
    SAFE = False; traceback.print_exc()
    print("\n❌ PREFLIGHT FALHOU — não rodo a bateria. Manda esse erro pro Claude.")

results = []
if SAFE:
    deadline = time.time() + TIME_BUDGET_MIN*60
    for exp in BATTERY:
        if time.time() > deadline - 12*60:
            print(f"⏹ orçamento esgotado — não inicio {exp['name']}"); break
        try:
            r = run_experiment(exp, deadline)
            if r:
                results.append(r)
                json.dump(results, open(f'{DRIVE}/runs/BATERIA_parcial.json','w'), ensure_ascii=False, indent=1)
        except Exception as e:
            traceback.print_exc(); print(f"  ❌ {exp['name']}: {e}")
    print("\n\n" + "="*64 + "\n=== RESULTADOS DA BATERIA ===\n" + "="*64)
    lines = ["| exp | fonte | horas | steps | WER | min |", "|---|---|---|---|---|---|"]
    for r in sorted(results, key=lambda x: x['wer']):
        ln = f"| {r['name']} | {r['source']} | {r['hours']} | {r['steps']} | {r['wer']:.1%} | {r['min']:.0f} |"
        lines.append(ln); print(ln)
    if results:
        best = min(results, key=lambda x: x['wer'])
        note = f"\n🏆 MELHOR: {best['name']} (WER {best['wer']:.1%}) → BASE-PT do Estágio B (notebook 2)\n"
    else:
        note = "\n(nenhum experimento concluiu — ver erros)\n"
    print(note)
    open(f'{DRIVE}/runs/BATERIA_results.md','w').write("# Bateria de língua\n\n" + "\n".join(lines) + "\n" + note)
    print("📄 salvo: Drive/TTS-ptbr-data/runs/BATERIA_results.md")